In [21]:
!pip install -q streamlit pyjwt pyngrok

In [22]:
%%writefile app.py
"""
Infosys FreightQuote — Smart Logistics Quotation & Authentication Portal
Infosys Springboard Internship 7.0 | Batch 1 | Milestone 1

Streamlit + JWT + ngrok + Gmail OTP authentication module.

Run this from Google Colab (see the Milestone1.ipynb notebook in this folder).
All secrets (JWT_SECRET, NGROK_AUTHTOKEN, EMAIL_PASSWORD, EMAIL_ADDRESS) must be
stored in Colab Secrets — never hard-code them here.
"""

import os
import re
import json
import time
import jwt
import smtplib
import hashlib
import random
import string
from datetime import datetime, timedelta
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

import streamlit as st

# --------------------------------------------------------------------------------------
# CONFIG — secrets are read from environment variables that the Colab notebook injects
# from Colab Secrets. Never hard-code real values here.
# --------------------------------------------------------------------------------------
JWT_SECRET = os.environ.get("JWT_SECRET", "")
EMAIL_ADDRESS = os.environ.get("EMAIL_ADDRESS", "")
EMAIL_PASSWORD = os.environ.get("EMAIL_PASSWORD", "")

# Admin credentials are defined directly in code per Step 11 (NOT a signup account).
# Change these before you deploy / demo.
ADMIN_USERNAME = "admin"
ADMIN_PASSWORD = "Admin@123"

USERS_FILE = "users.json"
JWT_EXPIRY_MINUTES = 60
OTP_EXPIRY_MINUTES = 5

SECURITY_QUESTIONS = [
    "What is your favourite pet's name?",
    "What city were you born in?",
    "What is your mother's maiden name?",
    "What was the name of your first school?",
    "What is your favourite book?",
]

# --------------------------------------------------------------------------------------
# STORAGE HELPERS  (simple JSON file — good enough for a Colab demo / milestone)
# --------------------------------------------------------------------------------------
def load_users():
    if not os.path.exists(USERS_FILE):
        return {}
    with open(USERS_FILE, "r") as f:
        return json.load(f)


def save_users(users):
    with open(USERS_FILE, "w") as f:
        json.dump(users, f, indent=2)


def hash_value(value: str) -> str:
    return hashlib.sha256(value.encode()).hexdigest()


# --------------------------------------------------------------------------------------
# VALIDATION HELPERS  (Step 8)
# --------------------------------------------------------------------------------------
EMAIL_REGEX = re.compile(r"^[A-Za-z]{2,}[A-Za-z0-9._%+-]*@[A-Za-z]{2,}\.[A-Za-z]{2,}$")


def is_valid_email(email: str) -> bool:
    return bool(EMAIL_REGEX.match(email or ""))


def is_valid_password(pw: str) -> tuple[bool, str]:
    if not pw or len(pw) < 8:
        return False, "Password must be at least 8 characters long."
    if not re.search(r"[A-Z]", pw):
        return False, "Password must contain at least one uppercase letter."
    if not re.search(r"[a-z]", pw):
        return False, "Password must contain at least one lowercase letter."
    if not re.search(r"[0-9]", pw):
        return False, "Password must contain at least one number."
    if not re.search(r"[!@#$%^&*(),.?\":{}|<>_\-\[\]/\\+=~`;']", pw):
        return False, "Password must contain at least one special symbol."
    return True, ""


# --------------------------------------------------------------------------------------
# JWT HELPERS  (Step 7)
# --------------------------------------------------------------------------------------
def issue_jwt(username: str) -> str:
    payload = {
        "sub": username,
        "iat": datetime.utcnow(),
        "exp": datetime.utcnow() + timedelta(minutes=JWT_EXPIRY_MINUTES),
    }
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")


def verify_jwt(token: str):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        return payload["sub"]
    except Exception:
        return None


# --------------------------------------------------------------------------------------
# OTP + EMAIL HELPERS  (Step 9)
# --------------------------------------------------------------------------------------
def generate_otp() -> str:
    return "".join(random.choices(string.digits, k=6))


def send_otp_email(to_email: str, otp: str) -> bool:
    if not EMAIL_ADDRESS or not EMAIL_PASSWORD:
        st.error("Email is not configured. Ask the admin to set EMAIL_ADDRESS / EMAIL_PASSWORD secrets.")
        return False
    try:
        subject = "Infosys FreightQuote — Password Reset Verification"

        # Plain-text fallback (for clients that don't render HTML)
        text_body = f"""Hello,

Use the One-Time Password (OTP) below to reset your account password.

    {otp}

This OTP will expire in {OTP_EXPIRY_MINUTES} minutes.

This OTP is valid for a single use only. If you did not request a password reset,
you can safely ignore this email — your account remains secure.
"""

        # Styled HTML version — Deep Aurora theme, matches the app UI
        html_body = f"""\
<html>
  <body style="margin:0;padding:0;background-color:#0B0F2A;font-family:Arial,Helvetica,sans-serif;">
    <table role="presentation" width="100%" cellpadding="0" cellspacing="0"
           style="background:linear-gradient(135deg,#0B0F2A,#171338,#0A2233);padding:36px 0;">
      <tr>
        <td align="center">
          <table role="presentation" width="480" cellpadding="0" cellspacing="0"
                 style="background-color:rgba(255,255,255,0.045);border-radius:16px;overflow:hidden;
                        border:1px solid rgba(139,92,246,0.30);box-shadow:0 12px 32px rgba(0,0,0,0.4);">

            <!-- Header -->
            <tr>
              <td align="center" style="background-color:#0B0F2A;border-bottom:2px solid #22D3EE;padding:26px 20px 22px 20px;">
                <div style="color:#22D3EE;font-size:11px;font-weight:800;letter-spacing:2px;
                            text-transform:uppercase;margin-bottom:10px;">Security Notice</div>
                <div style="color:#F8FAFC;font-size:19px;font-weight:800;">
                  Infosys Intelligent Freight Quote
                </div>
                <div style="color:#A5B4CB;font-size:12px;margin-top:4px;letter-spacing:.3px;">
                  Password Reset Verification
                </div>
              </td>
            </tr>

            <!-- Body -->
            <tr>
              <td style="padding:30px 34px 10px 34px;">
                <p style="color:#F1F5F9;font-size:15px;margin:0 0 4px 0;">Hello,</p>
                <p style="color:#A5B4CB;font-size:14px;line-height:1.5;margin:0 0 22px 0;">
                  Use the One-Time Password (OTP) below to reset your account password.
                </p>
              </td>
            </tr>

            <!-- OTP box -->
            <tr>
              <td align="center" style="padding:0 34px 8px 34px;">
                <div style="background:linear-gradient(90deg,rgba(34,211,238,0.14),rgba(139,92,246,0.14));
                            border:1px solid rgba(34,211,238,0.35);border-radius:12px;
                            padding:18px 24px;display:inline-block;min-width:220px;">
                  <span style="font-size:32px;font-weight:800;letter-spacing:8px;color:#22D3EE;">
                    {otp}
                  </span>
                </div>
              </td>
            </tr>

            <!-- Expiry note -->
            <tr>
              <td align="center" style="padding:16px 34px 6px 34px;">
                <div style="background-color:rgba(255,255,255,0.06);border-radius:8px;padding:10px 16px;
                            display:inline-block;color:#A5B4CB;font-size:12.5px;">
                  This OTP will expire in {OTP_EXPIRY_MINUTES} minutes
                </div>
              </td>
            </tr>

            <!-- Footer note -->
            <tr>
              <td style="padding:22px 34px 30px 34px;">
                <hr style="border:none;border-top:1px solid rgba(139,92,246,0.2);margin:0 0 16px 0;">
                <p style="color:#6B7A99;font-size:12px;line-height:1.5;margin:0;">
                  This OTP is valid for a single use only. If you did not request a password
                  reset, you can safely ignore this email — your account remains secure.
                </p>
              </td>
            </tr>

            <!-- Brand strip -->
            <tr>
              <td align="center" style="background-color:#0B0F2A;padding:12px;">
                <span style="color:#6B7A99;font-size:11px;letter-spacing:.4px;">
                  Infosys Springboard 7.0 · FreightQuote Authentication Portal
                </span>
              </td>

            </tr>

          </table>
        </td>
      </tr>
    </table>
  </body>
</html>
"""


        msg = MIMEMultipart("alternative")
        msg["Subject"] = subject
        msg["From"] = EMAIL_ADDRESS
        msg["To"] = to_email
        msg.attach(MIMEText(text_body, "plain"))
        msg.attach(MIMEText(html_body, "html"))

        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as server:
            server.login(EMAIL_ADDRESS, EMAIL_PASSWORD)
            server.sendmail(EMAIL_ADDRESS, to_email, msg.as_string())
        return True
    except Exception as e:
        st.error(f"Could not send OTP email: {e}")
        return False


# --------------------------------------------------------------------------------------
# PAGE CONFIG + THEME  ("Deep Aurora" — indigo/teal gradient with glowing accents)
# --------------------------------------------------------------------------------------
st.set_page_config(page_title="Infosys FreightQuote", page_icon="📦", layout="centered")

CUSTOM_CSS = """
<style>
    /* Palette — Deep Aurora:
       background : #0B0F2A -> #10163B -> #0A1F2E (indigo-to-teal night gradient)
       card       : glassy panel with a soft violet/teal glow border
       accent     : #22D3EE cyan-teal (primary), #8B5CF6 violet (secondary/hover)
       text       : #F1F5F9 (primary), #A5B4CB (muted)
    */
    @keyframes auroraDrift {
        0%   { background-position: 0% 0%; }
        50%  { background-position: 100% 100%; }
        100% { background-position: 0% 0%; }
    }

    .stApp {
        background: linear-gradient(135deg, #0B0F2A, #171338, #0A2233, #0B0F2A);
        background-size: 280% 280%;
        animation: auroraDrift 24s ease infinite;
        color: #F1F5F9;
    }
    #MainMenu, footer, header {visibility: hidden;}

    div[data-testid="stHorizontalBlock"] button {
        border-radius: 10px !important;
        font-weight: 700 !important;
        white-space: nowrap !important;
        padding: 0.55rem 1rem !important;
        transition: transform 0.15s ease, box-shadow 0.15s ease, background-color 0.15s ease;
    }

    .fq-title {
        text-align: center;
        font-size: 2.4rem;
        font-weight: 900;
        margin-bottom: 0px;
        color: #F8FAFC;
        letter-spacing: 0.3px;
        text-shadow: 0 0 22px rgba(34,211,238,0.35), 0 0 40px rgba(139,92,246,0.20);
    }
    .fq-title::after {
        content: "";
        display: block;
        width: 70px;
        height: 3px;
        margin: 10px auto 0 auto;
        border-radius: 3px;
        background: linear-gradient(90deg, #22D3EE, #8B5CF6);
    }
    .fq-subtitle {
        text-align: center;
        color: #A5B4CB;
        margin-top: 14px;
        margin-bottom: 30px;
        font-size: 0.95rem;
        letter-spacing: 0.3px;
    }
    .fq-section-header {
        font-size: 1.35rem;
        font-weight: 800;
        margin-bottom: 18px;
        color: #F1F5F9;
        border-bottom: 2px solid rgba(139,92,246,0.35);
        padding-bottom: 10px;
    }
    .fq-card {
        background: rgba(255,255,255,0.045);
        border: 1px solid rgba(139,92,246,0.25);
        border-radius: 16px;
        padding: 28px 30px;
        margin-bottom: 18px;
        box-shadow: 0 10px 34px rgba(0,0,0,0.35), 0 0 0 1px rgba(34,211,238,0.05) inset;
    }
    /* Style Streamlit's native bordered container (st.container(border=True))
       to match our card look — used instead of manual HTML divs, which don't
       actually nest widgets and were causing empty boxes to appear. */
    div[data-testid="stVerticalBlockBorderWrapper"] {
        background: rgba(255,255,255,0.045) !important;
        border: 1px solid rgba(139,92,246,0.25) !important;
        border-radius: 16px !important;
        box-shadow: 0 10px 34px rgba(0,0,0,0.35);
        margin-bottom: 18px;
    }
    .fq-welcome-box {
        background: linear-gradient(90deg, rgba(34,211,238,0.14), rgba(139,92,246,0.14));
        border: 1px solid rgba(34,211,238,0.35);
        border-radius: 12px;
        padding: 16px 18px;
        margin: 14px 0 18px 0;
        font-weight: 700;
        color: #F1F5F9;
    }
    .fq-footer {
        text-align: center;
        color: #6B7A99;
        font-size: 0.8rem;
        margin-top: 40px;
    }

    /* Text + select inputs: crisp light fields, dark ink text, subtle violet border */
    div[data-testid="stTextInput"] input,
    div[data-testid="stSelectbox"] div[data-baseweb="select"] {
        background-color: #F8FAFC !important;
        color: #0F172A !important;
        border-radius: 9px !important;
        border: 1px solid #3B3268 !important;
    }
    label, .stMarkdown, .stRadio label, .stCheckbox label {
        color: #E2E8F0 !important;
    }

    /* Primary buttons: cyan-to-violet gradient with glow, lifts on hover */
    .stButton>button {
        background: linear-gradient(90deg, #22D3EE, #8B5CF6);
        color: #0B0F2A;
        border: none;
        border-radius: 10px;
        padding: 9px 22px;
        font-weight: 800;
        box-shadow: 0 6px 18px rgba(34,211,238,0.25);
    }
    .stButton>button:hover {
        background: linear-gradient(90deg, #8B5CF6, #F472B6);
        color: #ffffff;
        transform: translateY(-2px);
        box-shadow: 0 10px 24px rgba(139,92,246,0.35);
    }

    /* Navbar row buttons: deep glass pills, glow border on hover */
    div[data-testid="stHorizontalBlock"] button {
        background-color: rgba(255,255,255,0.06) !important;
        color: #F1F5F9 !important;
        border: 1px solid rgba(139,92,246,0.30) !important;
    }
    div[data-testid="stHorizontalBlock"] button:hover {
        background-color: rgba(34,211,238,0.10) !important;
        color: #22D3EE !important;
        border: 1px solid #22D3EE !important;
        transform: translateY(-2px);
    }

    /* Alerts recolored to fit the deep aurora theme */
    div[data-testid="stAlert"] {
        border-radius: 12px !important;
    }
</style>
"""
st.markdown(CUSTOM_CSS, unsafe_allow_html=True)

# --------------------------------------------------------------------------------------
# SESSION STATE INIT
# --------------------------------------------------------------------------------------
defaults = {
    "page": "login",
    "jwt_token": None,
    "username": None,
    "admin_logged_in": False,
    "fp_stage": "choose",       # choose -> security / otp flows
    "fp_username_verified": None,
    "otp_value": None,
    "otp_expiry": None,
    "otp_email": None,
}
for k, v in defaults.items():
    if k not in st.session_state:
        st.session_state[k] = v

# Hidden admin entry point — no button in the UI. Reach it by opening:
#   https://<your-ngrok-url>/?admin=1
if st.query_params.get("admin") and st.session_state.page != "admin":
    st.session_state.page = "admin"


def go(page):
    st.session_state.page = page
    st.session_state.fp_stage = "choose"
    st.rerun()


# --------------------------------------------------------------------------------------
# NAVBAR  (Login / Signup / Forgot Password) — hidden once logged in
# Admin is intentionally NOT shown here. It's reached only via a hidden URL flag:
#   https://your-ngrok-url/?admin=1
# --------------------------------------------------------------------------------------
def render_navbar():
    cols = st.columns(3)
    labels = [("Login", "login"), ("Signup", "signup"),
              ("Forgot Password", "forgot")]
    for col, (label, target) in zip(cols, labels):
        with col:
            if st.button(label, key=f"nav_{target}", use_container_width=True):
                go(target)


def render_header():
    st.markdown('<div class="fq-title">Infosys FreightQuote</div>', unsafe_allow_html=True)
    st.markdown(
        '<div class="fq-subtitle">Smart Logistics Quotation &amp; Authentication Portal</div>',
        unsafe_allow_html=True,
    )


def render_dashboard_navbar():
    cols = st.columns([2, 5, 2])
    with cols[0]:
        if st.button("Dashboard", use_container_width=True):
            go("dashboard")
    with cols[2]:
        if st.button("Logout", use_container_width=True):
            st.session_state.jwt_token = None
            st.session_state.username = None
            go("login")


# --------------------------------------------------------------------------------------
# LOGIN PAGE
# --------------------------------------------------------------------------------------
def page_login():
    render_navbar()
    render_header()

    with st.container(border=True):
        st.markdown('<div class="fq-section-header">User Login</div>', unsafe_allow_html=True)

        with st.form("login_form"):
            identifier = st.text_input("Username / Email *")
            password = st.text_input("Password *", type="password")
            remember = st.checkbox("Remember Me")
            submitted = st.form_submit_button("Login")

        if submitted:
            if not identifier or not password:
                st.error("Please fill in all mandatory fields.")
            else:
                users = load_users()
                user = users.get(identifier) or next(
                    (u for u in users.values() if u.get("email") == identifier), None
                )
                if user and user["password_hash"] == hash_value(password):
                    token = issue_jwt(user["username"])
                    st.session_state.jwt_token = token
                    st.session_state.username = user["username"]
                    st.success("Login successful!")
                    time.sleep(0.4)
                    go("dashboard")
                else:
                    st.error("Invalid username/email or password.")

    st.markdown(
        '<div class="fq-footer">Developed for Infosys Springboard Internship 7.0 · Batch 1 · Milestone 1</div>',
        unsafe_allow_html=True,
    )


# --------------------------------------------------------------------------------------
# SIGNUP PAGE
# --------------------------------------------------------------------------------------
def page_signup():
    render_navbar()
    render_header()

    with st.container(border=True):
        st.markdown('<div class="fq-section-header">Create Account</div>', unsafe_allow_html=True)

        with st.form("signup_form"):
            username = st.text_input("Username *")
            email = st.text_input("Email *")
            password = st.text_input("Password *", type="password")
            confirm_password = st.text_input("Confirm Password *", type="password")
            security_question = st.selectbox("Security Question *", SECURITY_QUESTIONS)
            security_answer = st.text_input("Security Answer *")
            submitted = st.form_submit_button("Sign Up")

        if submitted:
            if not all([username, email, password, confirm_password, security_answer]):
                st.error("Please fill in all mandatory fields.")
                return
            if not is_valid_email(email):
                st.error("Please enter a valid email address (e.g. ab@cd.ef).")
                return
            ok, msg = is_valid_password(password)
            if not ok:
                st.error(msg)
                return
            if password != confirm_password:
                st.error("Password and Confirm Password do not match.")
                return

            users = load_users()
            if username in users:
                st.error("This username is already taken. Please choose another.")
                return

            users[username] = {
                "username": username,
                "email": email,
                "password_hash": hash_value(password),
                "security_question": security_question,
                "security_answer_hash": hash_value(security_answer.strip().lower()),
            }
            save_users(users)
            st.success("Account created successfully! Please login.")
            time.sleep(0.6)
            go("login")


# --------------------------------------------------------------------------------------
# FORGOT PASSWORD PAGE  (Security Question route + OTP route)
# --------------------------------------------------------------------------------------
def page_forgot():
    render_navbar()
    render_header()

    with st.container(border=True):
        st.markdown('<div class="fq-section-header">Forgot Password</div>', unsafe_allow_html=True)

        route = st.radio("Choose a recovery method:", ["Security Question", "Email OTP"], horizontal=True)
        st.divider()

        users = load_users()

        # ---------------- SECURITY QUESTION ROUTE ----------------
        if route == "Security Question":
            if st.session_state.fp_stage != "security_verified":
                with st.form("sq_verify_form"):
                    username = st.text_input("Username *")
                    submitted = st.form_submit_button("Get Security Question")
                if submitted:
                    if username in users:
                        st.session_state.fp_username_verified = username
                        st.session_state.fp_stage = "security_question_shown"
                    else:
                        st.error("No account found with that username.")

            if st.session_state.fp_stage == "security_question_shown":
                username = st.session_state.fp_username_verified
                question = users[username]["security_question"]
                with st.form("sq_answer_form"):
                    st.info(f"Security Question: {question}")
                    answer = st.text_input("Your Answer *")
                    submitted = st.form_submit_button("Verify Answer")
                if submitted:
                    if hash_value(answer.strip().lower()) == users[username]["security_answer_hash"]:
                        st.session_state.fp_stage = "security_verified"
                        st.success("Answer verified. Please set a new password below.")
                    else:
                        st.error("Incorrect answer. Please try again.")

            if st.session_state.fp_stage == "security_verified":
                username = st.session_state.fp_username_verified
                with st.form("sq_reset_form"):
                    new_pw = st.text_input("New Password *", type="password")
                    confirm_pw = st.text_input("Confirm New Password *", type="password")
                    submitted = st.form_submit_button("Reset Password")
                if submitted:
                    ok, msg = is_valid_password(new_pw)
                    if not ok:
                        st.error(msg)
                    elif new_pw != confirm_pw:
                        st.error("Passwords do not match.")
                    else:
                        users[username]["password_hash"] = hash_value(new_pw)
                        save_users(users)
                        st.success("Password reset successful! Please login.")
                        time.sleep(0.6)
                        go("login")

        # ---------------- OTP ROUTE ----------------
        else:
            if st.session_state.fp_stage != "otp_verified":
                with st.form("otp_request_form"):
                    email = st.text_input("Registered Email *")
                    submitted = st.form_submit_button("Send OTP")
                if submitted:
                    match = next((u for u in users.values() if u.get("email") == email), None)
                    if not match:
                        st.error("No account found with that email.")
                    else:
                        otp = generate_otp()
                        if send_otp_email(email, otp):
                            st.session_state.otp_value = otp
                            st.session_state.otp_expiry = datetime.utcnow() + timedelta(minutes=OTP_EXPIRY_MINUTES)
                            st.session_state.otp_email = email
                            st.session_state.fp_username_verified = match["username"]
                            st.session_state.fp_stage = "otp_sent"
                            st.success("OTP sent to your email.")

            if st.session_state.fp_stage == "otp_sent":
                with st.form("otp_verify_form"):
                    otp_input = st.text_input("Enter OTP *")
                    submitted = st.form_submit_button("Verify OTP")
                if submitted:
                    if datetime.utcnow() > st.session_state.otp_expiry:
                        st.error("OTP expired. Please request a new one.")
                        st.session_state.fp_stage = "choose"
                    elif otp_input == st.session_state.otp_value:
                        st.session_state.fp_stage = "otp_verified"
                        st.success("OTP verified. Please set a new password below.")
                    else:
                        st.error("Incorrect OTP.")

            if st.session_state.fp_stage == "otp_verified":
                username = st.session_state.fp_username_verified
                with st.form("otp_reset_form"):
                    new_pw = st.text_input("New Password *", type="password")
                    confirm_pw = st.text_input("Confirm New Password *", type="password")
                    submitted = st.form_submit_button("Reset Password")
                if submitted:
                    ok, msg = is_valid_password(new_pw)
                    if not ok:
                        st.error(msg)
                    elif new_pw != confirm_pw:
                        st.error("Passwords do not match.")
                    else:
                        users[username]["password_hash"] = hash_value(new_pw)
                        save_users(users)
                        st.success("Password reset successful! Please login.")
                        time.sleep(0.6)
                        go("login")


# --------------------------------------------------------------------------------------
# USER DASHBOARD  (Step 10)
# --------------------------------------------------------------------------------------
def page_dashboard():
    username = verify_jwt(st.session_state.jwt_token) if st.session_state.jwt_token else None
    if not username:
        st.warning("Your session has expired. Please login again.")
        time.sleep(0.6)
        go("login")
        return

    render_dashboard_navbar()
    render_header()
    st.markdown(
        f'<div class="fq-footer" style="margin-top:0;margin-bottom:20px;">'
        f'Developed by {username} | Infosys Springboard 7.0</div>',
        unsafe_allow_html=True,
    )

    with st.container(border=True):
        st.markdown('<div class="fq-section-header">User Dashboard</div>', unsafe_allow_html=True)
        st.markdown(f'<div class="fq-welcome-box">Welcome, {username}</div>', unsafe_allow_html=True)
        st.write("You have successfully logged in.")
        st.success("JWT Authentication Successful.")

        users = load_users()
        user = users.get(username, {})
        st.markdown("#### Account Details")
        st.write(f"**Username:** {username}")
        st.write(f"**Email:** {user.get('email', '-')}")


# --------------------------------------------------------------------------------------
# ADMIN PAGE  (Step 11)
# --------------------------------------------------------------------------------------
def page_admin():
    render_navbar()
    render_header()

    with st.container(border=True):
        if not st.session_state.admin_logged_in:
            st.markdown('<div class="fq-section-header">Admin Login</div>', unsafe_allow_html=True)
            with st.form("admin_login_form"):
                admin_user = st.text_input("Admin Username *")
                admin_pass = st.text_input("Admin Password *", type="password")
                submitted = st.form_submit_button("Login as Admin")
            if submitted:
                if admin_user == ADMIN_USERNAME and admin_pass == ADMIN_PASSWORD:
                    st.session_state.admin_logged_in = True
                    st.rerun()
                else:
                    st.error("Invalid admin credentials.")
        else:
            st.markdown('<div class="fq-section-header">Admin Dashboard — Registered Users</div>', unsafe_allow_html=True)
            users = load_users()
            if not users:
                st.info("No users registered yet.")
            else:
                rows = [{"Username": u["username"], "Email": u["email"]} for u in users.values()]
                st.table(rows)
            if st.button("Logout of Admin"):
                st.session_state.admin_logged_in = False
                st.rerun()


# --------------------------------------------------------------------------------------
# ROUTER
# --------------------------------------------------------------------------------------
page = st.session_state.page
if page == "login":
    page_login()
elif page == "signup":
    page_signup()
elif page == "forgot":
    page_forgot()
elif page == "dashboard":
    page_dashboard()
elif page == "admin":
    page_admin()
else:
    page_login()


Overwriting app.py


In [23]:
import os
from google.colab import userdata

os.environ["JWT_SECRET"] = userdata.get("JWT_SECRET")
os.environ["NGROK_AUTHTOKEN"] = userdata.get("NGROK_AUTHTOKEN")
os.environ["EMAIL_PASSWORD"] = userdata.get("EMAIL_PASSWORD")
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")

print("Secrets loaded into environment ✅")

Secrets loaded into environment ✅


In [24]:
from pyngrok import ngrok

ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
print("ngrok authenticated ✅")

ngrok authenticated ✅


In [25]:
import subprocess, time

# Kill any previous streamlit process
!pkill -f streamlit || true
time.sleep(2)

# Start streamlit in the background
process = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"]
)
time.sleep(6)

public_url = ngrok.connect(8501)
print("Your app is live at:", public_url)

^C


Your app is live at: NgrokTunnel: "https://retold-unopposed-reforest.ngrok-free.dev" -> "http://localhost:8501"
